# BiRRT vs. OMPL motion planning

Both `BiRRTPlanner` and `OMPLPlanner` satisfy the same `MotionPlanner` interface and plan over the same `ConfigurationSpace`, so we can swap one for the other. This notebook plans the Panda arm around an obstacle with each, compares planning time, and renders the resulting plans side by side.

Run top to bottom (`make_panda` compiles IKFast on first use).

In [ ]:
import time

import numpy as np
import pybullet as p
from spatialmath import SE3

from prpl_kinematics.collision import PyBulletCollisionChecker
from prpl_kinematics.geometry.shapes import BoxShape
from prpl_kinematics.planning import BiRRTPlanner, OMPLPlanner
from prpl_kinematics.robots import make_panda
from prpl_kinematics.tree.joints import FixedJoint
from prpl_kinematics.tree.kinematic_tree import Edge, Node

# A Panda with a box obstacle in front of it; the arm swings joint 1 around it.
robot = make_panda()
block = BoxShape(size=(0.1, 0.1, 0.5))
robot.tree.add_node(Node("obstacle", visuals=[block], collisions=[block]))
robot.tree.add_edge(
    Edge(robot.tree.root, "obstacle", FixedJoint(name="ofix", origin=SE3(0.45, 0.0, 0.6)))
)

checker = PyBulletCollisionChecker(p.connect(p.DIRECT))
checker.load(robot.tree)
checker.ignore(robot.allowed_collision_pairs)

arm = robot.groups["arm"]
start = robot.home
goal = {**dict(start), "panda_joint1": [1.2]}
assert not checker.in_collision(start) and not checker.in_collision(goal)

## Planning time

Plan with each planner several times (varying the random seed) and compare the mean wall-clock time and path length. `BiRRTPlanner` wraps `prpl_utils.BiRRT`; `OMPLPlanner` wraps OMPL's `RRTConnect` and simplifies the result.

In [ ]:
def benchmark(make_planner, trials=5):
    times, lengths, paths = [], [], []
    for seed in range(trials):
        planner = make_planner(np.random.default_rng(seed))
        t0 = time.perf_counter()
        path = planner.plan(start, goal)
        times.append(time.perf_counter() - t0)
        assert path is not None and all(not checker.in_collision(c) for c in path)
        lengths.append(len(path))
        paths.append(path)
    return times, lengths, paths


birrt_times, birrt_lengths, birrt_paths = benchmark(
    lambda rng: BiRRTPlanner(arm, checker.in_collision, rng, num_iters=1000)
)
ompl_times, ompl_lengths, ompl_paths = benchmark(
    lambda rng: OMPLPlanner(arm, checker.in_collision, rng, timeout=5.0)
)

print(f"{'planner':<8} {'mean time (ms)':>16} {'mean path length':>18}")
print(f"{'BiRRT':<8} {1000 * np.mean(birrt_times):>16.1f} {np.mean(birrt_lengths):>18.1f}")
print(f"{'OMPL':<8} {1000 * np.mean(ompl_times):>16.1f} {np.mean(ompl_lengths):>18.1f}")

## Render the plans

Render each plan as an inline animation. The renderer uses its own PyBullet client (separate from the collision checker, so the checker's bodies are not drawn over the robot).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

from prpl_kinematics.visualization import CameraParams, PyBulletRenderer, render_configurations

renderer = PyBulletRenderer(p.connect(p.DIRECT))
renderer.load(robot.tree)
camera = CameraParams(target=(0.2, 0.1, 0.6), distance=1.6, yaw=70.0, pitch=-20.0)


def plan_video(label, path):
    images = render_configurations(renderer, path, camera)
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.set_title(label)
    ax.axis("off")
    canvas = ax.imshow(images[0])

    def update(i):
        canvas.set_data(images[i])
        return [canvas]

    anim = animation.FuncAnimation(fig, update, frames=len(images), interval=60, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


display(plan_video("BiRRT", birrt_paths[0]))
display(plan_video("OMPL", ompl_paths[0]))

## A harder problem: a narrow passage

The easy scene favors `BiRRTPlanner` — its pure-Python core has low overhead, while `OMPLPlanner` pays a fixed setup cost. But on a hard problem the picture flips. Here an XY gantry must pass through a thin gap in a wall: BiRRT's uniform sampling rarely lands in the gap, so it wastes many iterations, while OMPL's `RRTConnect` threads it efficiently.

In [ ]:
from prpl_kinematics.tree.joints import PrismaticJoint
from prpl_kinematics.tree.kinematic_tree import KinematicTree
from prpl_kinematics.planning import JointSpace

GAP = 0.18  # robot is 0.1 wide, so ~0.04 clearance on each side
gantry = KinematicTree()
gantry.add_node(Node("jx"))
gantry.add_node(Node("gantry_robot", collisions=[BoxShape(size=(0.1, 0.1, 0.1))]))
gantry.add_node(Node("wall_left", collisions=[BoxShape(size=(2.5 - GAP / 2 + 1, 0.1, 0.1))]))
gantry.add_node(Node("wall_right", collisions=[BoxShape(size=(2.5 - GAP / 2, 0.1, 0.1))]))
gantry.add_edge(Edge("world", "jx", PrismaticJoint(name="gx", axis=(1, 0, 0), lower=-1, upper=5)))
gantry.add_edge(Edge("jx", "gantry_robot", PrismaticJoint(name="gy", axis=(0, 1, 0), lower=-1, upper=5)))
gantry.add_edge(Edge("world", "wall_left", FixedJoint(name="wlf", origin=SE3((-1 + 2.5 - GAP / 2) / 2, 2.5, 0))))
gantry.add_edge(Edge("world", "wall_right", FixedJoint(name="wrf", origin=SE3((2.5 + GAP / 2 + 5) / 2, 2.5, 0))))

gantry_checker = PyBulletCollisionChecker(p.connect(p.DIRECT))
gantry_checker.load(gantry)
gspace = JointSpace(gantry, ["gx", "gy"])
gstart, ggoal = {"gx": [0.0], "gy": [0.0]}, {"gx": [4.0], "gy": [4.0]}


def gantry_benchmark(make_planner, trials=5):
    times, paths = [], []
    for seed in range(trials):
        t0 = time.perf_counter()
        path = make_planner(np.random.default_rng(seed)).plan(gstart, ggoal)
        times.append(time.perf_counter() - t0)
        assert path is not None
        paths.append(path)
    return times, paths


g_birrt_times, g_birrt_paths = gantry_benchmark(
    lambda rng: BiRRTPlanner(gspace, gantry_checker.in_collision, rng, num_iters=2000)
)
g_ompl_times, g_ompl_paths = gantry_benchmark(
    lambda rng: OMPLPlanner(gspace, gantry_checker.in_collision, rng, timeout=10.0)
)
print(f"narrow passage, gap width {GAP}")
print(f"BiRRT mean {1000 * np.mean(g_birrt_times):>6.0f} ms")
print(
    f"OMPL  mean {1000 * np.mean(g_ompl_times):>6.0f} ms"
    f"  ({np.mean(g_birrt_times) / np.mean(g_ompl_times):.0f}x faster than BiRRT)"
)

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(6, 6))
ax.add_patch(mpatches.Rectangle((-1, 2.45), 2.5 - GAP / 2 + 1, 0.1, color="0.6"))
ax.add_patch(mpatches.Rectangle((2.5 + GAP / 2, 2.45), 2.5 - GAP / 2, 0.1, color="0.6"))
for path, color, label in [
    (g_birrt_paths[0], "tab:blue", "BiRRT"),
    (g_ompl_paths[0], "tab:orange", "OMPL"),
]:
    ax.plot([c["gx"][0] for c in path], [c["gy"][0] for c in path], color=color, lw=2, label=label)
ax.plot(0, 0, "go", markersize=10, label="start")
ax.plot(4, 4, "r*", markersize=16, label="goal")
ax.set_xlim(-1.2, 5.2)
ax.set_ylim(-1.2, 5.2)
ax.set_aspect("equal")
ax.legend(loc="lower right")
ax.set_title(f"Narrow passage (gap = {GAP})")
fig.tight_layout()